In [4]:

# PRACTICAL 7
# Analysis and Visualization of Stock Market Data

# 1. Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# ARIMA and time series analysis libraries
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_style("whitegrid")


# 2. Load the dataset

df = pd.read_csv("Stock_Prices.csv")

print("First five records:")
display(df.head())

print("\nDataset shape:", df.shape)

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe())

print("\nMissing values:")
print(df.isnull().sum())


# 3. Convert Date column into datetime format

# Identify the date column
date_col = next(
    (col for col in df.columns if col.lower().strip() in ["date", "datetime", "timestamp"]),
    None
)

if date_col is None:
    raise ValueError("Date column not found. Check the CSV column names.")

df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

# Remove rows with invalid dates
df.dropna(subset=[date_col], inplace=True)

# Sort data according to date
df.sort_values(by=date_col, inplace=True)

# Set date as index
df.set_index(date_col, inplace=True)

print("\nDataset after date conversion:")
display(df.head())


# Identify closing price and volume columns

def find_column(possible_names):
    for col in df.columns:
        if col.lower().strip() in possible_names:
            return col
    return None

close_col = find_column([
    "close", "closing price", "close price", "adj close", "adj_close"
])

volume_col = find_column([
    "volume", "trading volume"
])

if close_col is None:
    raise ValueError("Closing price column not found. Check the CSV column names.")

# Convert price and volume to numeric
df[close_col] = pd.to_numeric(df[close_col], errors="coerce")

if volume_col:
    df[volume_col] = pd.to_numeric(df[volume_col], errors="coerce")

df.dropna(subset=[close_col], inplace=True)

# If multiple records exist for the same date, aggregate them
df = df[~df.index.duplicated(keep="last")]

print("\nSelected closing price column:", close_col)
print("Selected volume column:", volume_col)


# 4. Visualize historical stock price trends

plt.figure(figsize=(14, 6))
plt.plot(df.index, df[close_col], label="Closing Price")
plt.title("Historical Stock Price Trend")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 5. Calculate and plot moving averages

df["Moving_Average_7"] = df[close_col].rolling(window=7).mean()
df["Moving_Average_30"] = df[close_col].rolling(window=30).mean()

plt.figure(figsize=(14, 6))

plt.plot(df.index, df[close_col], label="Closing Price", alpha=0.5)
plt.plot(df.index, df["Moving_Average_7"], label="7-Day Moving Average")
plt.plot(df.index, df["Moving_Average_30"], label="30-Day Moving Average")

plt.title("Stock Price with Moving Averages")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 6. Seasonality analysis

# Resample data into monthly average closing prices
monthly_data = df[close_col].resample("ME").mean()

plt.figure(figsize=(14, 6))
plt.plot(monthly_data.index, monthly_data.values, marker="o")
plt.title("Monthly Average Stock Price")
plt.xlabel("Month")
plt.ylabel("Average Closing Price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Monthly average prices by calendar month
monthly_seasonality = df[close_col].groupby(df.index.month).mean()

plt.figure(figsize=(10, 5))
monthly_seasonality.plot(kind="bar")
plt.title("Average Stock Price by Month")
plt.xlabel("Month")
plt.ylabel("Average Closing Price")
plt.tight_layout()
plt.show()

# Seasonal decomposition
# Use a period of 5 for approximate weekly patterns in daily trading data.
# Use 12 for monthly observations with yearly seasonality.

if len(df) >= 10:
    try:
        decomposition = seasonal_decompose(
            df[close_col].dropna(),
            model="additive",
            period=5
        )

        decomposition.plot()
        plt.suptitle("Seasonal Decomposition of Stock Prices")
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print("Seasonal decomposition could not be performed:", e)


# 7. Correlation analysis between stock prices and other variables

numeric_data = df.select_dtypes(include=np.number)

if len(numeric_data.columns) > 1:

    correlation = numeric_data.corr()

    plt.figure(figsize=(10, 7))
    sns.heatmap(
        correlation,
        annot=True,
        cmap="coolwarm",
        fmt=".2f"
    )

    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()

else:
    print("Not enough numeric columns for correlation analysis.")

# Plot closing price against trading volume
if volume_col:
    plt.figure(figsize=(10, 6))
    plt.scatter(df[volume_col], df[close_col], alpha=0.5)
    plt.title("Closing Price vs Trading Volume")
    plt.xlabel("Trading Volume")
    plt.ylabel("Closing Price")
    plt.tight_layout()
    plt.show()

    print(
        "Correlation between closing price and volume:",
        df[close_col].corr(df[volume_col])
    )


# 8. Stock price forecasting using ARIMA

# Use closing price time series
stock_data = df[close_col].dropna()

# Split data into training and testing sets
split_index = int(len(stock_data) * 0.8)

train = stock_data.iloc[:split_index]
test = stock_data.iloc[split_index:]

print("\nTraining records:", len(train))
print("Testing records:", len(test))

# Fit ARIMA model
# ARIMA(p,d,q) = (5,1,0)

model = ARIMA(train, order=(5, 1, 0))
model_fit = model.fit()

print("\nARIMA Model Summary:")
print(model_fit.summary())

# Forecast stock prices for testing period
forecast = model_fit.forecast(steps=len(test))

# Plot actual vs predicted prices
plt.figure(figsize=(14, 6))

plt.plot(train.index, train, label="Training Data")
plt.plot(test.index, test, label="Actual Stock Price")
plt.plot(test.index, forecast, label="Predicted Stock Price")

plt.title("Stock Price Forecast using ARIMA")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Evaluate model performance
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))

print("\nARIMA Model Performance")
print("Mean Absolute Error (MAE):", mae)
print("Root Mean Squared Error (RMSE):", rmse)


# Forecast future stock prices

future_steps = 10

future_forecast = model_fit.forecast(steps=future_steps)

print("\nFuture Stock Price Forecast:")
display(future_forecast)


# Plot future forecast

plt.figure(figsize=(14, 6))

plt.plot(stock_data.index, stock_data, label="Historical Stock Price")
plt.plot(
    pd.date_range(
        start=stock_data.index[-1],
        periods=future_steps + 1,
        freq="B"
    )[1:],
    future_forecast.values,
    marker="o",
    label="Future Forecast"
)

plt.title("Future Stock Price Prediction using ARIMA")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.tight_layout()
plt.show()




First five records:


,Price,Close,High,Low,Open,Volume
0,Ticker,BEL.NS,BEL.NS,BEL.NS,BEL.NS,BEL.NS
1,Date,NaN,NaN,NaN,NaN,NaN
2,2012-01-02,10.757936477661133,10.93730810375916,10.666858300644368,10.778219630865577,1220472
3,2012-01-03,11.044294357299805,11.187871559035488,10.746799361241878,10.976284064319463,910008
4,2012-01-04,11.253098487854004,11.35571136451063,10.950036252453096,10.950036252453096,2594493



Dataset shape: (3205, 6)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3205 entries, 0 to 3204
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Price   3205 non-null   object
 1   Close   3204 non-null   object
 2   High    3204 non-null   object
 3   Low     3204 non-null   object
 4   Open    3204 non-null   object
 5   Volume  3204 non-null   object
dtypes: object(6)
memory usage: 150.4+ KB

Statistical summary:


,Price,Close,High,Low,Open,Volume
count,3205,3204,3204,3204,3204,3204
unique,3205,3056,3202,3199,3204,3196
top,Ticker,44.54867172241211,21.32113309258351,51.17658605345742,BEL.NS,1607364
freq,1,3,2,2,1,2



Missing values:
Price     0
Close     1
High      1
Low       1
Open      1
Volume    1
dtype: int64


ValueError: Date column not found. Check the CSV column names.